# Reclassify scaleup queries with GPT-5.5 → Gold query file

Reclassifies every `query_text` in `data/scaleup/queries/scaleup_queries_v2.parquet`
(250 queries) into a **clean, single-valued, controlled-vocabulary** taxonomy using
**OpenAI GPT-5.5** (`gpt-5.5-2026-04-23`, the project's `chatgpt` engine), and writes
the result to **`data/scaleup/queries/reclass_queries.parquet`** — the **Gold** query
file used for all downstream analysis.

**Output columns:** `query_id, query_text, complexity, technicality, sensitivity,
intent, domain, content_type, response_type, temporality`.

**Method:** Responses API `responses.parse` with a Pydantic schema of `Literal`
enums → the API guarantees in-vocabulary values. **No web_search tool** (pure
parametric classification). One query per call, checkpointed to a JSONL so re-runs
resume. URL/website queries are forced to `response_type=navigation`, `domain=link`
(GPT instruction + regex safety-net); their other 6 facets are still classified.

> Determinism caveat: GPT-5.5 has no temperature/seed control on this model
> generation (see `thesis_config.py`); `model_served` is logged per call. Single pass (k=1).

In [1]:
# --- Bootstrap: project root on path, load .env (OPENAI_API_KEY), imports ---
import sys, json, re, time
from pathlib import Path
from datetime import datetime, timezone
from typing import Literal, get_args

# Locate project root (this notebook lives in notebooks/test/).
ROOT = Path.cwd()
while not (ROOT / "thesis_config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "thesis_config.py").exists(), f"could not find project root from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

from src.env import load_env
load_env()  # populate os.environ from .env (project convention; see scripts/test_openai_adapter.py)

import os
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY", "").strip(), "OPENAI_API_KEY not set (check .env)"
print("project root :", ROOT)
print("openai key   : SET")

project root : /Users/ganenthraravindran/Desktop/Thesis Data Pilot
openai key   : SET


In [2]:
# --- Config: model, paths, run knobs ---------------------------------------
import thesis_config as cfg

MODEL = cfg.ENGINES["chatgpt"]["model"]          # "gpt-5.5-2026-04-23"
REASONING_EFFORT = "low"                          # classification of short queries is easy; matches project default

QUERIES_DIR = ROOT / "data" / "scaleup" / "queries"
REGISTRY    = QUERIES_DIR / "scaleup_queries_v2.parquet"   # source (raw GeoBench tags)
OUT_PARQUET = QUERIES_DIR / "reclass_queries.parquet"      # Gold output
CKPT_JSONL  = QUERIES_DIR / "reclass_queries.jsonl"        # per-query checkpoint (resume-safe)
MANIFEST    = QUERIES_DIR / "reclass_queries_manifest.json"

assert REGISTRY.exists(), f"source registry not found: {REGISTRY}"
print("model        :", MODEL)
print("source       :", REGISTRY.relative_to(ROOT))
print("gold output  :", OUT_PARQUET.relative_to(ROOT))
print("checkpoint   :", CKPT_JSONL.relative_to(ROOT))

model        : gpt-5.5-2026-04-23
source       : data/scaleup/queries/scaleup_queries_v2.parquet
gold output  : data/scaleup/queries/reclass_queries.parquet
checkpoint   : data/scaleup/queries/reclass_queries.jsonl


In [3]:
# --- Controlled vocabulary + structured-output schema ----------------------
# Each facet is a Literal enum. responses.parse(text_format=...) turns these into a
# strict JSON schema, so the model can ONLY return in-vocabulary values (no cleaning).

Complexity   = Literal["simple", "intermediate", "complex"]
Technicality = Literal["technical", "non-technical"]
Sensitivity  = Literal["sensitive", "non-sensitive"]
Intent       = Literal["informational", "navigational", "transactional"]

# Domain: the 30 manifest domains (kept identical for comparability) + "link" for URLs.
Domain = Literal[
    "science", "arts and entertainment", "health", "jobs and education",
    "people and society", "food and drink", "computers and electronics", "history",
    "law and government", "business and industrial", "politics", "finance", "sports",
    "books and literature", "physics", "environment", "biology",
    "internet and telecom", "technology", "travel", "geography", "law",
    "pets and animals", "automotive", "philosophy", "religion", "engineering",
    "games", "language", "military", "link",
]

ContentType  = Literal["factual", "opinion", "debate", "mixed"]
ResponseType = Literal[
    "factual_answer", "explanation", "instruction", "comparison", "list",
    "recommendation", "prediction", "definition", "navigation",
]
Temporality  = Literal["evergreen", "latest", "historical", "undefined"]


class QueryFacets(BaseModel):
    """Single-valued classification of one search query."""
    complexity:    Complexity   = Field(description="simple = one fact/lookup; intermediate = some synthesis; complex = multi-step reasoning or many interacting parts.")
    technicality:  Technicality = Field(description="technical = needs domain/expert vocabulary or specialist knowledge; otherwise non-technical.")
    sensitivity:   Sensitivity  = Field(description="sensitive = health, finance, legal, political, safety, or personal-data topics where wrong answers cause real harm; else non-sensitive.")
    intent:        Intent       = Field(description="informational = seeking knowledge; navigational = trying to reach a specific site/page/login; transactional = wants to do/buy/download/use a tool.")
    domain:        Domain       = Field(description="THE single most suitable topic. Pick the most specific applicable category. Use 'link' ONLY when query_text is itself a URL/web address/path.")
    content_type:  ContentType  = Field(description="Epistemic nature of the answer: factual = objective verifiable fact; opinion = subjective/advice; debate = genuinely contested with no consensus; mixed = combines factual + subjective.")
    response_type: ResponseType = Field(description="Mode of response the query calls for: factual_answer, explanation, instruction (how-to/steps), comparison, list, recommendation, prediction, definition, or navigation (when query_text is a URL/web address to go to).")
    temporality:   Temporality  = Field(description="evergreen = answer stable over time; latest = needs current/recent info; historical = about the past; undefined = no clear time dependence.")

print("schema fields:", list(QueryFacets.model_fields))
print("domains      :", len(get_args(Domain)), "values (incl. 'link')")

schema fields: ['complexity', 'technicality', 'sensitivity', 'intent', 'domain', 'content_type', 'response_type', 'temporality']
domains      : 31 values (incl. 'link')


In [4]:
# --- Prompt, URL safety-net, and the single-query classify() call ----------
client = OpenAI()  # reads OPENAI_API_KEY from environment

INSTRUCTIONS = (
    "You are a meticulous search-query classifier. Given ONE search query, assign exactly one "
    "value per facet from the provided schema. Classify the query itself (the user's information "
    "need), not any particular answer. Be decisive; pick the single best label.\n"
    "Special rule: if the query_text is ITSELF a website / URL / domain / login path "
    "(e.g. 'www.example.com', 'http://x.org/', 'altavista.com', 'account/microsoft/login'), set "
    "domain='link' and response_type='navigation', and classify the remaining facets normally "
    "(such queries are intent='navigational')."
)

# Conservative regex backstop: clear URL markers, or a no-space host/path. This UNION's with the
# model's own judgement so obvious URLs are never missed (idempotent with the prompt rule above).
_URL_RE = re.compile(r"^(https?://|www\.)|\.(com|org|net|edu|gov|io|me|co|biz|info|us|uk|ca)(/|$)", re.I)

def looks_like_url(text: str) -> bool:
    s = (text or "").strip()
    if _URL_RE.search(s):
        return True
    if " " not in s and "/" in s:   # e.g. 'account/microsoft/login'
        return True
    return False

def classify(query_text: str) -> dict:
    """Classify one query. Returns {facets..., model_served}. No web_search tool attached."""
    resp = client.responses.parse(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=query_text,
        text_format=QueryFacets,
        reasoning={"effort": REASONING_EFFORT},
    )
    facets = resp.output_parsed.model_dump()
    facets["model_served"] = getattr(resp, "model", "") or ""
    return facets

# Smoke-test on the two example URL queries + one normal query before the full run.
for q in ["www.profedcreditunion.com", "what is the capital of France"]:
    print(f"{q!r:40s} url_regex={looks_like_url(q)} -> {classify(q)}")

'www.profedcreditunion.com'              url_regex=True -> {'complexity': 'simple', 'technicality': 'non-technical', 'sensitivity': 'sensitive', 'intent': 'navigational', 'domain': 'link', 'content_type': 'factual', 'response_type': 'navigation', 'temporality': 'undefined', 'model_served': 'gpt-5.5-2026-04-23'}


'what is the capital of France'          url_regex=False -> {'complexity': 'simple', 'technicality': 'non-technical', 'sensitivity': 'non-sensitive', 'intent': 'informational', 'domain': 'geography', 'content_type': 'factual', 'response_type': 'factual_answer', 'temporality': 'evergreen', 'model_served': 'gpt-5.5-2026-04-23'}


In [5]:
# --- Run: classify all 250 queries, checkpointing to JSONL (resume-safe) ----
src = pd.read_parquet(REGISTRY)[["query_id", "query_text"]].copy()
print(f"loaded {len(src)} queries from {REGISTRY.name}")

# Resume: load any query_ids already classified in a previous (partial) run.
done = {}
if CKPT_JSONL.exists():
    for line in CKPT_JSONL.read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            done[r["query_id"]] = r
    print(f"resuming: {len(done)} already in checkpoint")

todo = src[~src["query_id"].isin(done.keys())]
print(f"to classify this run: {len(todo)}")

t0 = time.time()
with CKPT_JSONL.open("a") as fh:
    for n, (_, row) in enumerate(todo.iterrows(), 1):
        qid, qtext = row["query_id"], row["query_text"]
        try:
            rec = {"query_id": qid, "query_text": qtext, **classify(qtext)}
        except Exception as e:
            print(f"  [ERROR] {qid} {qtext[:50]!r}: {e}")
            raise
        fh.write(json.dumps(rec) + "\n")
        fh.flush()
        done[qid] = rec
        if n % 25 == 0 or n == len(todo):
            print(f"  {n}/{len(todo)}  ({time.time()-t0:.0f}s)  last: {qtext[:48]!r}")

print(f"done. total classified: {len(done)}/{len(src)}  in {time.time()-t0:.0f}s")

loaded 250 queries from scaleup_queries_v2.parquet
to classify this run: 250


  25/250  (90s)  last: "What is, or should be, 'global history'?"


  50/250  (176s)  last: 'what are cytoskeleton in biology'


  75/250  (252s)  last: 'Given a <span> in HTML I want to change its colo'


  100/250  (348s)  last: 'grade to percentage calculator'


...

  150/250  (509s)  last: 'Should electroconvulsive therapy be used as a me'


  175/250  (601s)  last: 'how can data analytics and AI be used to combat '


  200/250  (684s)  last: 'what does the word comet mean'


  225/250  (768s)  last: 'what does globalization mean'


  250/250  (863s)  last: 'spiegel home catalog shopping online'
done. total classified: 250/250  in 863s


In [6]:
# --- Assemble in registry order, apply URL safety-net override, validate ----
FACETS = ["complexity", "technicality", "sensitivity", "intent",
          "domain", "content_type", "response_type", "temporality"]
VOCAB = {
    "complexity": Complexity, "technicality": Technicality, "sensitivity": Sensitivity,
    "intent": Intent, "domain": Domain, "content_type": ContentType,
    "response_type": ResponseType, "temporality": Temporality,
}

gold = src.copy()  # preserves the registry's query order
recs = gold["query_id"].map(done)
for f in FACETS:
    gold[f] = [r[f] for r in recs]
gold["model_served"] = [r.get("model_served", "") for r in recs]

# Regex safety-net: force URL/website queries to navigation + link (union with GPT's rule).
is_url = gold["query_text"].map(looks_like_url)
gold.loc[is_url, "response_type"] = "navigation"
gold.loc[is_url, "domain"] = "link"
print(f"URL/website queries forced to navigation+link: {int(is_url.sum())}")

# --- validation ---
assert len(gold) == len(src) == 250, f"row count {len(gold)} != 250"
assert gold["query_id"].is_unique, "duplicate query_id"
assert set(gold["query_id"]) == set(src["query_id"]), "query_id set drift vs registry"
for f in FACETS:
    allowed = set(get_args(VOCAB[f]))
    bad = set(gold[f].unique()) - allowed
    assert not bad, f"{f}: out-of-vocab values {bad}"
    assert gold[f].notna().all(), f"{f}: has nulls"
# Every URL query must be navigation AND link
urlrows = gold[is_url]
assert (urlrows["response_type"] == "navigation").all() and (urlrows["domain"] == "link").all()
print("VALIDATION PASSED: 250 rows, all facets in-vocab, no nulls, URL overrides applied.")
gold[["query_id", "query_text"] + FACETS].head(10)

URL/website queries forced to navigation+link: 6
VALIDATION PASSED: 250 rows, all facets in-vocab, no nulls, URL overrides applied.


,query_id,query_text,complexity,technicality,sensitivity,intent,domain,content_type,response_type,temporality
0,gb_1a77ad01ab570671,when does elena turn into a vampire in the tv ...,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
1,gb_314f7fd2ce040788,who played the oldest brother in 7th heaven,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
2,gb_eaab9147ac75176b,which came first the walking dead comic or show,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,historical
3,gb_51f95e9b9f4f6183,who plays unis in she's the man,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
4,gb_f3a09c028168dc20,where does the sound come from when you crack ...,simple,non-technical,sensitive,informational,health,factual,explanation,evergreen
5,gb_57e198d356ad745f,the book of the thousand nights and one night ...,simple,non-technical,non-sensitive,informational,books and literature,factual,factual_answer,evergreen
6,gb_588d144b9e0f9869,who sings the christmas song all i want for ch...,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
7,gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,historical
8,gb_a33bd88b6b1ff82a,when will the next episode of flash be aired,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,latest
9,gb_a61c19b7ea5345d6,only fools and horses del falls through the ba...,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen


In [7]:
# --- QC: per-facet distributions + spot-check vs original GeoBench tags ------
for f in FACETS:
    print(f"\n=== {f} ===")
    print(gold[f].value_counts().to_string())

# Spot-check: GPT facets next to the original (noisy) tags for 10 queries.
orig = pd.read_parquet(REGISTRY)[["query_id", "tags"]]
orig["tags"] = orig["tags"].map(lambda a: ", ".join(list(a)))
chk = gold.merge(orig, on="query_id")
print("\n\n=== spot-check: GPT vs original tags (10 rows) ===")
for _, r in chk.head(10).iterrows():
    print(f"\nQ: {r['query_text'][:70]}")
    print(f"   GPT : intent={r['intent']} | domain={r['domain']} | complexity={r['complexity']} | "
          f"content={r['content_type']} | response={r['response_type']} | temporality={r['temporality']}")
    print(f"   tags: {r['tags']}")


=== complexity ===
complexity
intermediate    121
simple          116
complex          13

=== technicality ===
technicality
non-technical    216
technical         34

=== sensitivity ===
sensitivity
non-sensitive    161
sensitive         89

=== intent ===
intent
informational    212
transactional     23
navigational      15

=== domain ===
domain
arts and entertainment       24
health                       24
politics                     16
jobs and education           16
food and drink               14
finance                      14
history                      13
law and government           12
language                     10
people and society           10
biology                       9
computers and electronics     9
science                       8
sports                        8
link                          7
technology                    6
books and literature          5
physics                       5
business and industrial       5
geography                     5
automoti

In [8]:
# --- Write the Gold parquet + manifest -------------------------------------
out = gold[["query_id", "query_text"] + FACETS].copy()  # exactly the 10 requested columns
out.to_parquet(OUT_PARQUET, index=False)

manifest = {
    "artifact": "reclass_queries_gold",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_registry": str(REGISTRY.relative_to(ROOT)),
    "classifier_model": MODEL,
    "models_served": sorted(set(gold["model_served"]) - {""}),
    "reasoning_effort": REASONING_EFFORT,
    "web_search": False,
    "n_queries": int(len(out)),
    "url_navigation_count": int(is_url.sum()),
    "columns": list(out.columns),
    "vocabularies": {f: list(get_args(VOCAB[f])) for f in FACETS},
}
MANIFEST.write_text(json.dumps(manifest, indent=2))

print("WROTE:")
print(" ", OUT_PARQUET.relative_to(ROOT), f"({OUT_PARQUET.stat().st_size:,} bytes, {len(out)} rows)")
print(" ", MANIFEST.relative_to(ROOT))
print("\nThis is the GOLD query file for all downstream analysis.")
print("Note: data/scaleup/** is gitignored -> use `git add -f` if you want to commit it.")
out.head()

WROTE:
  data/scaleup/queries/reclass_queries.parquet (22,476 bytes, 250 rows)
  data/scaleup/queries/reclass_queries_manifest.json

This is the GOLD query file for all downstream analysis.
Note: data/scaleup/** is gitignored -> use `git add -f` if you want to commit it.


,query_id,query_text,complexity,technicality,sensitivity,intent,domain,content_type,response_type,temporality
0,gb_1a77ad01ab570671,when does elena turn into a vampire in the tv ...,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
1,gb_314f7fd2ce040788,who played the oldest brother in 7th heaven,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
2,gb_eaab9147ac75176b,which came first the walking dead comic or show,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,historical
3,gb_51f95e9b9f4f6183,who plays unis in she's the man,simple,non-technical,non-sensitive,informational,arts and entertainment,factual,factual_answer,evergreen
4,gb_f3a09c028168dc20,where does the sound come from when you crack ...,simple,non-technical,sensitive,informational,health,factual,explanation,evergreen
